# Playground

Сбор сырых спортивных и киберспортивных событий с Polymarket.
Разбор в котировки пока не делаем: сначала смотрим, что вообще приходит.

Перед первым запуском, из корня проекта:

```
python -m venv .venv
.venv\Scripts\activate
pip install -e ".[dev]"
```

Дальше ячейки по порядку. Короткий прогон и полный — разные ячейки, полную не запускай случайно: она качает все лиги и пишет большой файл.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from arbx.config import PROJECT_ROOT
from arbx.logging_setup import setup_logging
from arbx.sources.polymarket import collect_raw, market_type_counts, preview_markets, preview_rows
from arbx.storage.files import latest_raw, load_raw, save_raw

setup_logging("INFO")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)
PROJECT_ROOT

## Короткий прогон

`max_seconds=10` — сколько успеет за 10 секунд, столько и вернёт. Крупные лиги (NBA, EPL, CS2, ...) идут первыми, так что в выборке будут живые матчи, а не пустые серии.

Даже короткий прогон получается тяжёлым: приоритетные лиги за ~10 секунд дают около тысячи событий и файл порядка 150 МБ, потому что у баскетбола на матч сотни рынков и мы сохраняем их целиком.

Можно вместо времени поставить `max_events=20`, или сузить до одной лиги: `series_codes=["cs2"]`.

In [ ]:
snap = collect_raw(max_seconds=10)
path = save_raw(snap)

print("file:", path)
print("reason:", snap.meta["stopped_reason"])
print("events:", snap.meta["event_count"], "markets:", snap.meta["market_count"])
print("sports:", snap.meta["sports_events"], "esports:", snap.meta["esports_events"])
print("series with events:", snap.meta["series_fetched"])
print("empty series so far:", snap.meta["series_empty"], "of", snap.meta["series_total"])

## Что собралось

Таблица ниже — смотровая, не формат хранения. На диске лежит сырой JSON: каталог `/sports` и события Gamma как есть.

In [ ]:
events = pd.DataFrame(preview_rows(snap))
events

In [ ]:
pd.DataFrame(market_type_counts(snap))

## Один событие изнутри

Поменять индекс, если хочется другое. `sportsMarketType` и `line` — это то, чем Gamma сам размечает рынок. Пока только смотрим.

In [ ]:
event = snap.payload["events"][0]
print(event["title"])
print("sport:", event.get("sport", {}).get("sport"), "| markets:", len(event.get("markets") or []))
print("tags:", [tag.get("slug") for tag in event.get("tags") or []])
pd.DataFrame(preview_markets(event, limit=25))

## Одна лига целиком

Удобно, когда надо посмотреть устройство конкретной серии, не дожидаясь всех 400+ лиг.

In [ ]:
one = collect_raw(series_codes=["cs2"])
print(one.meta["event_count"], "events,", one.meta["market_count"], "markets")
pd.DataFrame(preview_rows(one))

## Полный прогон

Все лиги из каталога `/sports`, и обычный спорт, и киберспорт. Занимает несколько минут. Файл может весить сотни мегабайт: у NBA на один матч приходится несколько сотен рынков, и они сохраняются целиком.

Повторный запуск не перезаписывает прошлый снимок, кладёт новый файл рядом.

In [ ]:
full = collect_raw()
full_path = save_raw(full)
print("file:", full_path)
print("reason:", full.meta["stopped_reason"])
print("events:", full.meta["event_count"], "markets:", full.meta["market_count"])
print("sports:", full.meta["sports_events"], "esports:", full.meta["esports_events"])
print("series with events:", len(full.meta["series_fetched"]))
print("empty series:", full.meta["series_empty"])
print("errors:", full.meta["series_errors"])

## Открыть последний снимок без сети

In [ ]:
saved = latest_raw("polymarket")
print(saved)
again = load_raw(saved)
print(again.meta["event_count"], "events from", again.fetched_at.isoformat())
pd.DataFrame(preview_rows(again)).head(20)